In [1]:
# Version 10
# should calculate with float64 for precision
import json
import pandas as pd
import gc
from pathlib import Path
import numpy as np

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

# The t-ohlcv data was pre-fetched
folder_path = find_project_root() / "data" / "bigData"

symbol = "BTCUSDT"
tf = "5m"

PATH_5m =folder_path / f"{symbol}-raw-{tf}.json"

with open(PATH_5m) as f:
    arr = json.load(f)

# Binance API fetched columns
cols = ["timestamp", "open", "high", "low", "close", "vol", "close-timestamp", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol","checksum"]

df_5 = pd.DataFrame(arr, columns=cols)
del arr  
gc.collect() # free the Python var, preserve RAM

# Halve memory: float64 -> float32 for price/volume columns
df_5[["open", "high", "low", "close", "vol", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol"]] = df_5[["open", "high", "low", "close", "vol", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol"]].astype("float32")

print(f"5m Shape: {df_5.shape} | Memory: {df_5.memory_usage(deep=True).sum()/ 1e6:.2f} MB") # ~23 MB

# sort by timestamp
df_5 = df_5.sort_values('timestamp').reset_index(drop=True)
df_5.head()

5m Shape: (229771, 12) | Memory: 23.44 MB


,timestamp,open,high,low,close,vol,close-timestamp,quote-vol,taker#,taker-buy-basevol,taker-buy-quotevol,checksum
0,1703910600000,42006.898438,42026.980469,41978.000000,42014.410156,67.633591,1703910899999,2840850.500,2548.0,36.562511,1.535670e+06,0
1,1703910900000,42014.421875,42037.109375,41994.000000,42030.011719,50.828709,1703911199999,2135396.500,1963.0,27.786961,1.167242e+06,0
2,1703911200000,42030.000000,42054.800781,42029.019531,42031.988281,43.327000,1703911499999,1821481.750,1680.0,20.556490,8.641578e+05,0
3,1703911500000,42032.000000,42032.000000,42004.000000,42004.011719,24.898029,1703911799999,1046175.250,1621.0,7.909600,3.323482e+05,0
4,1703911800000,42004.000000,42004.011719,41973.921875,41973.941406,44.289219,1703912099999,1859286.375,1864.0,16.972891,7.124834e+05,0


In [2]:
# 24 features on single timeframe was almost usable
# integrate 15m as Higher timeframe
FIFTEEN_MIN_MS = 900_000
FORTYFIVE_MIN_MS = 2_700_000  # 45 * 60 * 1000

mtf = "15m"
PATH_15m =folder_path / f"{symbol}-raw-{mtf}.json"
with open(PATH_15m) as f:
    harr = json.load(f)

mtf_load = pd.DataFrame(harr, columns=cols)

# get only t-ohlcv
df_15 = mtf_load[["timestamp", "open", "high", "low", "close", "vol"]].copy()

# Cast to float — Binance JSON stores OHLCV as strings; without this:
#   "sum" concatenates strings, "max"/"min" use lexicographic order (silently wrong)
df_15[["open", "high", "low", "close", "vol"]] = df_15[["open", "high", "low", "close", "vol"]].astype("float64")

df_15["ts_45m"] = (df_15["timestamp"] // FORTYFIVE_MIN_MS) * FORTYFIVE_MIN_MS

# also create 45m as High time frame
# --- Aggregate ---
df_45 = df_15.copy().groupby("ts_45m", sort=True).agg(
    open  = ("open",  "first"),
    high  = ("high",  "max"),
    low   = ("low",   "min"),
    close = ("close", "last"),
    vol   = ("vol",   "sum"),
    bar_count = ("timestamp", "count"),
).reset_index().rename(columns={"ts_45m": "timestamp"})

# Flag incomplete bars (< 3 x 15m bars) — usually only first/last bar
incomplete = df_45[df_45["bar_count"] < 3]
if len(incomplete):
    print(f"Incomplete 45m bars (< 3 x 15m): {len(incomplete)} — dropping")
    df_45 = df_45[df_45["bar_count"] == 3]

df_45 = df_45.drop(columns=["bar_count"]).reset_index(drop=True)
# df_45 timestamps = first 15m bar in bucket; re-derive the 45m bucket key
df_45['ts_45m'] = (df_45['timestamp'] // FORTYFIVE_MIN_MS) * FORTYFIVE_MIN_MS
print(f"45m bars produced: {len(df_45):,}")
print(f"Expected ~{len(df_15) // 3:,} (15m ÷ 3)")

del harr, mtf_load 
gc.collect() # free the Python var, preserve RAM

Incomplete 45m bars (< 3 x 15m): 1 — dropping
45m bars produced: 25,641
Expected ~25,641 (15m ÷ 3)


0

In [3]:
# preview 15m data
df_15 = df_15.sort_values('timestamp').reset_index(drop=True)
df_15.head()

,timestamp,open,high,low,close,vol,ts_45m
0,1703610900000,42325.52,42350.00,42136.36,42258.64,627.36994,1703610900000
1,1703611800000,42258.63,42270.00,41637.60,41979.25,1922.77912,1703610900000
2,1703612700000,41979.25,42049.60,41771.53,42009.99,1271.82748,1703610900000
3,1703613600000,42010.00,42147.04,41966.48,42049.33,709.17414,1703613600000
4,1703614500000,42049.33,42094.17,41860.00,41886.87,478.50625,1703613600000


In [4]:
# preview 45m data
df_45 = df_45.sort_values('timestamp').reset_index(drop=True)
df_45.head()

,timestamp,open,high,low,close,vol,ts_45m
0,1703610900000,42325.52,42350.00,41637.60,42009.99,3821.97654,1703610900000
1,1703613600000,42010.00,42147.04,41832.42,41993.02,1675.78196,1703613600000
2,1703616300000,41993.01,42306.45,41993.01,42280.00,1305.30148,1703616300000
3,1703619000000,42279.99,42285.00,42055.00,42128.87,618.15150,1703619000000
4,1703621700000,42128.88,42234.04,42082.57,42135.13,589.33335,1703621700000


# Does price hit ±ATR42 within the next 3 bars?
- prepare ATR42 on 5m 
- used to make triple barrier

In [5]:
# ATR42 on 5m — equivalent to ATR14 on 15m in time

prev_close_5m = df_5['close'].shift(1)

# True range
tr_5m = pd.concat([
    df_5['high'] - df_5['low'],
    (df_5['high'] - prev_close_5m).abs(),
    (df_5['low']  - prev_close_5m).abs(),
], axis=1).max(axis=1)

# Average True Range : SMA or RMA
df_5['atr42'] = tr_5m.rolling(42).mean().astype('float32')

print(f"ATR42 range: {df_5['atr42'].min():.2f} – {df_5['atr42'].max():.2f}")
df_5[['timestamp', 'close', 'atr42']].head()

ATR42 range: 7.42 – 1552.81


,timestamp,close,atr42
0,1703910600000,42014.410156,NaN
1,1703910900000,42030.011719,NaN
2,1703911200000,42031.988281,NaN
3,1703911500000,42004.011719,NaN
4,1703911800000,41973.941406,NaN


# Triple Barrier
- Create ternary label on 5m

In [6]:
# triple-barrier on 5m (gap-aware)
FIVE_MIN_MS = 300_000

# Snap 5m timestamps to grid (absorbs ms jitter) — needed for gap detection
df_5['ts_5m'] = (df_5['timestamp'] // FIVE_MIN_MS) * FIVE_MIN_MS

# Triple-barrier on 5m (gap-aware)
ATR_MULT    = 1.0
MAX_BARS_5M = 3   # 1h lookahead — tune: 6=30m, 12=1h, 24=2h, 48=4h

# df_5   = ltf_df.reset_index(drop=True)

high_arr  = df_5['high'].values
low_arr   = df_5['low'].values
open_arr  = df_5['open'].values
close_arr = df_5['close'].values
atr_arr   = df_5['atr42'].values
ts5_arr   = df_5['ts_5m'].values

labels = np.full(len(df_5), np.nan)

for i in range(len(df_5)):
    if np.isnan(atr_arr[i]):
        continue

    entry = close_arr[i]
    tp    = atr_arr[i] * ATR_MULT
    sl    = atr_arr[i] * ATR_MULT
    label = 0  # default: timeout

    for k in range(1, MAX_BARS_5M + 1):
        j = i + k
        if j >= len(df_5):
            break
        # Stop at data gap — don't evaluate across market closures
        if ts5_arr[j] != ts5_arr[i] + k * FIVE_MIN_MS:
            break

        h, l, o = high_arr[j], low_arr[j], open_arr[j]
        hit_up   = (h - entry) >= tp
        hit_down = (entry - l) >= sl

        if hit_up and hit_down:
            # Both barriers in same bar: open proximity infers direction first
            label = 1 if abs(o - (entry + tp)) < abs(o - (entry - sl)) else -1
            break
        elif hit_up:
            label = 1
            break
        elif hit_down:
            label = -1
            break

    labels[i] = label

df_5['label'] = labels

# Print Stats
total  = df_5['label'].notna().sum()
n_up   = (df_5['label'] ==  1).sum()
n_down = (df_5['label'] == -1).sum()
n_time = (df_5['label'] ==  0).sum()
n_nan  = df_5['label'].isna().sum()

print(f"Total labeled  : {total:,}")
print(f"  Up   ( 1)    : {n_up:,}  ({n_up   / total * 100:.1f}%)")
print(f"  Down (-1)    : {n_down:,}  ({n_down / total * 100:.1f}%)")
print(f"  Timeout (0)  : {n_time:,}  ({n_time  / total * 100:.1f}%)  ← used Two-stage model if this Label=0 dominates") # > 50%
print(f"  NaN (warmup) : {n_nan:,}")

Total labeled  : 229,730
  Up   ( 1)    : 76,565  (33.3%)
  Down (-1)    : 78,157  (34.0%)
  Timeout (0)  : 75,008  (32.7%)  ← used Two-stage model if this Label=0 dominates
  NaN (warmup) : 41


In [7]:
# Drop no use cols
drop_cols = ["close-timestamp", "taker-buy-quotevol","checksum", "quote-vol"]
df_5.drop(columns=drop_cols, inplace=True)

# Special : Group N - ICT - Pivot point aggregate
skip this to save  = skip group N. The tasks are;
- Create label on 5m
- add str hunt on 15m and 45m
- fixed BoS error and price order error -> cause leakage in training data

In [8]:
def _fix_break_of_structure(isXXX, isSource, high, low):
    """
    Ensure isXXX alternates strictly between 1 and -1.
    On a break (1,1 or -1,-1), insert the best counter swing between the two offending positions,
    but ONLY if it is structurally valid:
      - Inserting a  1 (high): candidate high must be > max(low[p1],  low[p2])
      - Inserting a -1 (low) : candidate low  must be < min(high[p1], high[p2])
    If no valid candidate exists, remove the less extreme of the two duplicates instead.
    Returns (isXXX, insertions) where insertions = list of (inserted_idx, p2_idx).
    """
    insertions = []
    changed = True
    while changed:
        changed = False
        non_zero = np.where(isXXX != 0)[0]

        for i in range(1, len(non_zero)):
            p1, p2 = non_zero[i - 1], non_zero[i]
            v1, v2 = isXXX[p1], isXXX[p2]

            if v1 != v2:
                continue

            counter    = np.int8(-v1)
            candidates = np.arange(p1 + 1, p2)

            inserted = False
            if len(candidates) > 0:
                source_cands = candidates[isSource[candidates] == counter]
                pool = source_cands if len(source_cands) > 0 else candidates

                if counter == -1:
                    threshold  = min(high[p1], high[p2])
                    valid_pool = pool[low[pool] < threshold]
                else:
                    threshold  = max(low[p1], low[p2])
                    valid_pool = pool[high[pool] > threshold]

                if len(valid_pool) > 0:
                    best = (valid_pool[np.argmin(low[valid_pool])]
                            if counter == -1
                            else valid_pool[np.argmax(high[valid_pool])])
                    isXXX[best] = counter
                    insertions.append((int(best), int(p2)))
                    inserted = True

            if not inserted:
                if v1 == 1:
                    isXXX[p1 if high[p1] < high[p2] else p2] = 0
                else:
                    isXXX[p1 if low[p1]  > low[p2] else p2] = 0

            changed = True
            break

    return isXXX, insertions


def _fix_price_order(isXXX, high, low):
    """
    Fix price ordering violations by inserting 2 bridging marks instead of removing.
    Falls back to removal only if no valid bridge candidates exist.
    Returns (isXXX, insertions) where insertions = list of (inserted_idx, p2_idx).
    """
    insertions = []
    changed = True
    while changed:
        changed = False
        non_zero = np.where(isXXX != 0)[0]

        for i in range(1, len(non_zero)):
            p1, p2 = non_zero[i-1], non_zero[i]
            v1, v2 = isXXX[p1], isXXX[p2]

            if v1 == 1 and v2 == -1 and low[p2] >= high[p1]:
                cands = np.arange(p1 + 1, p2)
                valid_lows = cands[low[cands] < high[p1]]
                if len(valid_lows) > 0:
                    best_low = valid_lows[np.argmin(low[valid_lows])]
                    after = np.arange(best_low + 1, p2)
                    valid_highs = after[high[after] > low[best_low]]
                    if len(valid_highs) > 0:
                        best_high = valid_highs[np.argmax(high[valid_highs])]
                        isXXX[best_low]  = -1
                        isXXX[best_high] = 1
                        insertions.append((int(best_low),  int(p2)))
                        insertions.append((int(best_high), int(p2)))
                        changed = True
                        break
                isXXX[p2] = 0
                changed = True
                break

            elif v1 == -1 and v2 == 1 and high[p2] <= low[p1]:
                cands = np.arange(p1 + 1, p2)
                valid_highs = cands[high[cands] > low[p1]]
                if len(valid_highs) > 0:
                    best_high = valid_highs[np.argmax(high[valid_highs])]
                    after = np.arange(best_high + 1, p2)
                    valid_lows = after[low[after] < high[best_high]]
                    if len(valid_lows) > 0:
                        best_low = valid_lows[np.argmin(low[valid_lows])]
                        isXXX[best_high] = 1
                        isXXX[best_low]  = -1
                        insertions.append((int(best_high), int(p2)))
                        insertions.append((int(best_low),  int(p2)))
                        changed = True
                        break
                isXXX[p2] = 0
                changed = True
                break

    return isXXX, insertions


In [9]:
def validate_markers(df, col):
    """
    Validate a marker column (e.g. 'isITR', 'isLTR') across the entire DataFrame.

    Checks two invariants:
      1. BOS (Break-of-Structure): consecutive non-zero entries must alternate sign (no 1,1 or -1,-1).
      2. Price order: after a high (1), the next low (-1) must be strictly below it,
         and after a low (-1), the next high (1) must be strictly above it.

    Returns
    -------
    dict with keys:
        'bos_violations'   : DataFrame of rows where sign repeats
        'order_violations' : DataFrame of rows where price order is broken
        'is_valid'         : True only if both checks pass
    """
    rows = df[df[col] != 0][['timestamp', 'high', 'low', col]].copy()
    rows['price_at_mark'] = rows.apply(
        lambda r: r['high'] if r[col] == 1 else r['low'], axis=1
    )

    vals = rows[col].values

    # --- BOS check: consecutive same-sign entries ---
    bos_mask = [False] + [vals[i] == vals[i - 1] for i in range(1, len(vals))]
    bos_violations = rows[bos_mask]

    # --- Price order check ---
    order_fail_idx = []
    for i in range(1, len(rows)):
        prev = rows.iloc[i - 1]
        curr = rows.iloc[i]
        if curr[col] == 1:          # current is a high → must be above prev low
            if curr['high'] <= prev['low']:
                order_fail_idx.append(rows.index[i])
        else:                        # current is a low → must be below prev high
            if curr['low'] >= prev['high']:
                order_fail_idx.append(rows.index[i])
    order_violations = rows.loc[order_fail_idx]

    # --- Print summary ---
    label = col
    total = len(rows)
    print(f"=== {label} validation ({total} marks across full dataset) ===")

    if len(bos_violations):
        print(f"  BOS violations : {len(bos_violations)}")
        print(bos_violations[['timestamp', 'high', 'low', col, 'price_at_mark']].to_string())
    else:
        print("  BOS violations : 0  — alternation OK")

    if len(order_violations):
        print(f"  Order violations: {len(order_violations)}")
        print(order_violations[['timestamp', 'high', 'low', col, 'price_at_mark']].to_string())
    else:
        print("  Order violations: 0  — price ordering OK")

    is_valid = len(bos_violations) == 0 and len(order_violations) == 0
    print(f"  Valid: {is_valid}\n")

    return {
        'bos_violations':   bos_violations,
        'order_violations': order_violations,
        'is_valid':         is_valid,
    }


In [11]:
# STR features
def strHunt(df, window=4):
    n = len(df)
    df = df.copy()

    high = df['high'].values
    low  = df['low'].values

    isSTR = np.zeros(n, dtype=np.int8)

    for i in range(window, n - window):
        win_high = high[i - window : i + window + 1]
        win_low  = low [i - window : i + window + 1]

        is_swing_high = high[i] >= win_high.max()
        is_swing_low  = low[i]  <= win_low.min()

        if is_swing_high and not is_swing_low:
            isSTR[i] = 1
        elif is_swing_low and not is_swing_high:
            isSTR[i] = -1

    isSTR, bos_ins = _fix_break_of_structure(isSTR, isSTR, high, low)
    isSTR, po_ins  = _fix_price_order(isSTR, high, low)

    # inserted_p2: pivot_idx → p2_idx that caused the insertion
    inserted_p2 = dict(bos_ins + po_ins)

    # available_at: pivot_idx → first index where this mark is causally safe to use
    #   natural mark at i  → available at i + window (all confirmation bars closed)
    #   inserted mark at i → available at p2 + window (need p2 confirmed to know insertion exists)
    available_at = {}
    for i in np.where(isSTR != 0)[0]:
        available_at[int(i)] = inserted_p2.get(int(i), int(i)) + window

    str_confirmed = np.zeros(n, dtype=np.int8)
    for i, avail in available_at.items():
        if avail < n:
            str_confirmed[avail] = isSTR[i]

    return isSTR, str_confirmed, available_at


# add str_confirmed
isSTR_15, sc_15, avail_15 = strHunt(df_15, window=4)
df_15["isSTR"]         = isSTR_15
df_15["str_confirmed"] = sc_15

isSTR_45, sc_45, avail_45 = strHunt(df_45, window=4)
df_45["isSTR"]         = isSTR_45
df_45["str_confirmed"] = sc_45

validate_markers(df_15, 'isSTR')
validate_markers(df_45, 'isSTR')

marks = df_45['isSTR'].value_counts().to_dict()
print(f"45m shape: {df_45.shape}")
print(f"45m isSTR marks: {marks}  (highs: {marks.get(1,0)}, lows: {marks.get(-1,0)})")

marks = df_15['isSTR'].value_counts().to_dict()
print(f"15m shape: {df_15.shape}")
print(f"15m isSTR marks: {marks}  (highs: {marks.get(1,0)}, lows: {marks.get(-1,0)})")

=== isSTR validation (13715 marks across full dataset) ===
  BOS violations : 0  — alternation OK
  Order violations: 0  — price ordering OK
  Valid: True

=== isSTR validation (4776 marks across full dataset) ===
  BOS violations : 0  — alternation OK
  Order violations: 0  — price ordering OK
  Valid: True

45m shape: (25641, 9)
45m isSTR marks: {0: 20865, 1: 2388, -1: 2388}  (highs: 2388, lows: 2388)
15m shape: (76924, 9)
15m isSTR marks: {0: 63209, 1: 6858, -1: 6857}  (highs: 6858, lows: 6857)


In [12]:
def audit_no_lookahead(df, available_at, label, window=4):
    """
    Verify str_confirmed has zero look-ahead leakage.

    For every pivot i in available_at:
      - str_confirmed must fire at available_at[i], not earlier
      - available_at[i] >= i + window  (all confirmation bars must have closed)

    Prints a pass/fail summary with any violations listed.
    """
    violations = []

    for pivot_i, avail_j in available_at.items():
        # Rule 1: confirmation index must be >= pivot + window
        if avail_j < pivot_i + window:
            violations.append({
                'pivot': pivot_i, 'fires_at': avail_j,
                'min_allowed': pivot_i + window,
                'gap': (pivot_i + window) - avail_j,
                'reason': 'fires before confirmation bars closed'
            })

    total_marks = len(available_at)
    print(f"=== Look-ahead audit: {label} ({total_marks} marks) ===")
    if violations:
        print(f"  FAIL — {len(violations)} violation(s):")
        for v in violations[:10]:  # show first 10
            print(f"    pivot={v['pivot']}  fires_at={v['fires_at']}  "
                  f"min_allowed={v['min_allowed']}  ({v['reason']})")
        if len(violations) > 10:
            print(f"    ... and {len(violations)-10} more")
    else:
        print(f"  PASS — all {total_marks} marks fire at or after pivot + {window}")

    # Summary: how many are natural vs inserted (inserted fire later than i+window)
    natural  = sum(1 for i, j in available_at.items() if j == i + window)
    inserted = total_marks - natural
    print(f"  Natural marks : {natural}  (fired at pivot + {window})")
    print(f"  Inserted marks: {inserted}  (fired at p2 + {window}, delayed further)\n")
    return len(violations) == 0

audit_no_lookahead(df_15, avail_15, label="15m STR", window=4)
audit_no_lookahead(df_45, avail_45, label="45m STR", window=4)

=== Look-ahead audit: 15m STR (13715 marks) ===
  PASS — all 13715 marks fire at or after pivot + 4
  Natural marks : 11630  (fired at pivot + 4)
  Inserted marks: 2085  (fired at p2 + 4, delayed further)

=== Look-ahead audit: 45m STR (4776 marks) ===
  PASS — all 4776 marks fire at or after pivot + 4
  Natural marks : 3960  (fired at pivot + 4)
  Inserted marks: 816  (fired at p2 + 4, delayed further)



True

In [13]:
# Map str_confirmed to 5m timeframe
# One real risk to check: data gaps. 
# The timestamp arithmetic + FIFTEEN_MIN_MS assumes continuous data. 
# If there's a gap i.e. Friday -> Monday, the lookup key T_{i+4} + 15m still points to the correct next bucket for that bar

# str_confirmed[j] fires at 45m bucket T_j
# lookup key = T_j + 45m
# → first 5m bar at T_j + 45m = exactly when 45m bar j closes ✅


# STR_WINDOW        = 4
FIVE_MIN_MS = 300_000
FIFTEEN_MIN_MS    = 900_000
FORTY_FIVE_MIN_MS = 2_700_000

# str_confirmed is already built causally inside strHunt (available_at array)
# because price orderfix and BoS fixed created subtle leakage
# No shift needed here — just map to 5m

# Add bucket keys to 5m bars
df_5['ts_5m'] = (df_5['timestamp'] // FIVE_MIN_MS) * FIVE_MIN_MS
df_5['ts_15m'] = (df_5['timestamp'] // FIFTEEN_MIN_MS)    * FIFTEEN_MIN_MS
df_5['ts_45m'] = (df_5['timestamp'] // FORTY_FIVE_MIN_MS) * FORTY_FIVE_MIN_MS

# Build lookup: bucket_key → confirmed value
lookup_15m = {int(k) + FIFTEEN_MIN_MS:    v for k, v in df_15.set_index('timestamp')['str_confirmed'].to_dict().items()}
lookup_45m = {int(k) + FORTY_FIVE_MIN_MS: v for k, v in df_45.set_index('ts_45m')['str_confirmed'].to_dict().items()}

# Map to 5m — broad pass (all bars in bucket get the value)
df_5['15STR_confirmed'] = df_5['ts_15m'].map(lookup_15m).fillna(0).astype('int8')
df_5['45STR_confirmed'] = df_5['ts_45m'].map(lookup_45m).fillna(0).astype('int8')

# Keep only the FIRST 5m bar in each bucket (confirmation fires once, at bucket open)
is_first_in_15m = df_5.groupby('ts_15m').cumcount() == 0
is_first_in_45m = df_5.groupby('ts_45m').cumcount() == 0

df_5['15STR_confirmed'] = df_5['15STR_confirmed'].where(is_first_in_15m, 0)
df_5['45STR_confirmed'] = df_5['45STR_confirmed'].where(is_first_in_45m, 0)

print(f"15STR_confirmed  highs: {(df_5['15STR_confirmed']==1).sum():,}  lows: {(df_5['15STR_confirmed']==-1).sum():,}")
print(f"45STR_confirmed  highs: {(df_5['45STR_confirmed']==1).sum():,}  lows: {(df_5['45STR_confirmed']==-1).sum():,}")

15STR_confirmed  highs: 5,809  lows: 5,771
45STR_confirmed  highs: 1,997  lows: 1,947


In [15]:
df_15.head()

,timestamp,open,high,low,close,vol,ts_45m,isSTR,str_confirmed
0,1703610900000,42325.52,42350.00,42136.36,42258.64,627.36994,1703610900000,0,0
1,1703611800000,42258.63,42270.00,41637.60,41979.25,1922.77912,1703610900000,0,0
2,1703612700000,41979.25,42049.60,41771.53,42009.99,1271.82748,1703610900000,0,0
3,1703613600000,42010.00,42147.04,41966.48,42049.33,709.17414,1703613600000,0,0
4,1703614500000,42049.33,42094.17,41860.00,41886.87,478.50625,1703613600000,0,0


In [ ]:
# add last keylevel — split into keylv_high and keylv_low
#
# keylv_high: carries the highest high since the last confirmed swing HIGH
#             updates only when sig[i] == 1, range = high[last_high_i : i]
# keylv_low : carries the lowest low since the last confirmed swing LOW
#             updates only when sig[i] == -1, range = low[last_low_i : i]
# barsSince  : resets to 0 on either signal (high or low), counts bars since last confirmation
#
# Cold-start fill (before first signal fires):
#   keylv_high expands with running max of high[0..i-1]  → 0 null rows, fully causal
#   keylv_low  expands with running min of low[0..i-1]   → "highest/lowest seen so far"
#   Once first signal fires, plain carry-forward resumes as normal.

def add_keylv(df: pd.DataFrame, lv: str) -> pd.DataFrame:
    n    = len(df)
    high = df['high'].values
    low  = df['low'].values
    sig  = df[f"{lv}STR_confirmed"].values

    keylv_high = np.empty(n, dtype='float64')
    keylv_low  = np.empty(n, dtype='float64')
    bars_since = np.zeros(n, dtype='int32')

    # cold-start seed — no confirmed pivot yet
    keylv_high[0] = high[0]
    keylv_low[0]  = low[0]

    last_high_i = 0   # index of last swing HIGH confirmation
    last_low_i  = 0   # index of last swing LOW  confirmation
    high_init   = False  # True once first sig==1  fires
    low_init    = False  # True once first sig==-1 fires

    for i in range(1, n):
        if sig[i] == 1:
            # swing HIGH confirmed: extreme is max high since last high confirmation
            keylv_high[i] = round(float(high[last_high_i : i].max()), 4)
            keylv_low[i]  = keylv_low[i - 1]   # carry forward
            bars_since[i] = 0
            last_high_i   = i
            high_init     = True

        elif sig[i] == -1:
            # swing LOW confirmed: extreme is min low since last low confirmation
            keylv_low[i]  = round(float(low[last_low_i : i].min()), 4)
            keylv_high[i] = keylv_high[i - 1]  # carry forward
            bars_since[i] = 0
            last_low_i    = i
            low_init      = True

        else:
            # After init: plain carry-forward
            # Before init: expand running extreme — causal (uses bar i-1 only)
            keylv_high[i] = keylv_high[i-1] if high_init else max(keylv_high[i-1], high[i-1])
            keylv_low[i]  = keylv_low[i-1]  if low_init  else min(keylv_low[i-1],  low[i-1])
            bars_since[i] = bars_since[i-1] + 1   # purely backward-looking ✅

    df = df.copy()
    df[f"{lv}STR_keylv_high"] = keylv_high
    df[f"{lv}STR_keylv_low"]  = keylv_low
    df[f"barsSince{lv}STR"]   = bars_since
    return df

df_5 = add_keylv(df_5, "15")
df_5 = add_keylv(df_5, "45")
print(df_5.columns)
df_5.head()

In [ ]:
# Drop intermediate bucket-key columns (used only for mapping, not features)
drop_int_cols = ['ts_5m', 'ts_15m', 'ts_45m']
df_5.drop(columns=drop_int_cols, inplace=True)
print(f"Dropped: {drop_int_cols}")
print(f"Remaining columns ({len(df_5.columns)}): {df_5.columns.tolist()}")

In [ ]:
print(df_5.columns)
df_5.head()

# Save Data

In [ ]:
# Export marker to json visulize the data on Chart
# save export to jsonl
# Export marker to json visulize the data on Chart
# save export to jsonl

save_path = find_project_root() / "data" / "mlData" / "raw"
save_path.mkdir(parents=True, exist_ok=True)

out_path = save_path / "BTCUSDT-5m-v10.jsonl"
df_5.to_json(out_path, orient="records", lines=True)
print(f"Saved {len(df_5):,} rows → {out_path}")

In [ ]:
print(df_5.columns)